# Notebook 03 - Train model du doan gia nha (Ha Noi)

Muc tieu:
- Train baseline + model chinh tren du lieu clean.
- Danh gia theo MAE/RMSE/R2.
- Luu model tot nhat vao `webDemo/BackEnd/model1.joblib`.

In [2]:
%pip install numpy pandas scikit-learn joblib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\asus\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from joblib import dump

PROJECT_ROOT = Path("..").resolve()
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "housing_hanoi_clean.csv"
MODEL_PATH = PROJECT_ROOT / "webDemo" / "BackEnd" / "model1.joblib"

DATA_PATH, MODEL_PATH

(WindowsPath('C:/Users/asus/Desktop/DuDoanGiaNha/data/processed/housing_hanoi_clean.csv'),
 WindowsPath('C:/Users/asus/Desktop/DuDoanGiaNha/webDemo/BackEnd/model1.joblib'))

In [4]:
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head(3)

Shape: (356, 14)


,Gia,Gia_m2,dienTich,chieuNgang,chieuDai,Phongngu,PhongTam,SoTang,Loai,GiayTo,TinhTrangNoiThat,Phuong,Quan,list_id
0,9.000000e+09,2.571429e+08,35.0,350.0,10.0,3.0,4.0,5.0,"Nhà ngõ, hẻm",Đã có sổ,Khong ro,Phường Ngọc Lâm,Quận Long Biên,129628407
1,1.760000e+10,3.384615e+08,52.0,4.4,12.0,4.0,4.0,6.0,"Nhà mặt phố, mặt tiền",Đã có sổ,Nội thất cao cấp,Phường Long Biên,Quận Long Biên,130919547
2,6.500000e+09,2.124183e+08,30.6,7.0,4.3,3.0,4.0,5.0,"Nhà ngõ, hẻm",Đã có sổ,Nội thất đầy đủ,Phường Long Biên,Quận Long Biên,130571107


## 1) Chon feature dung theo backend

In [5]:
# Phai khop PredictorService.FEATURE_ORDER trong backend
FEATURES = [
    "chieuDai",
    "chieuNgang",
    "dienTich",
    "Phongngu",
    "SoTang",
    "PhongTam",
    "Loai",
    "GiayTo",
    "TinhTrangNoiThat",
    "Phuong",
]
TARGET = "Gia"

missing_features = [c for c in FEATURES if c not in df.columns]
if missing_features:
    raise ValueError(f"Thieu cot feature: {missing_features}")

X = df[FEATURES].copy()
y = pd.to_numeric(df[TARGET], errors="coerce")

valid_idx = y.notna()
X = X[valid_idx]
y = y[valid_idx]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (356, 10)
y shape: (356,)


In [6]:
numeric_features = ["chieuDai", "chieuNgang", "dienTich", "Phongngu", "SoTang", "PhongTam"]
categorical_features = ["Loai", "GiayTo", "TinhTrangNoiThat", "Phuong"]

numeric_transformer = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="median"))]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

## 2) Train/Test split + train RandomForest

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

rf_model = RandomForestRegressor(
    n_estimators=400,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1,
)

pipeline = Pipeline(steps=[("preprocessor", preprocessor), ("model", rf_model)])
pipeline.fit(X_train, y_train)

pred = pipeline.predict(X_test)
mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

# CV chi de tham khao on dinh
cv_neg_rmse = cross_val_score(
    pipeline,
    X_train,
    y_train,
    cv=5,
    scoring="neg_root_mean_squared_error",
)
cv_rmse_mean = -cv_neg_rmse.mean()

metrics_df = pd.DataFrame([
    {
        "model": "RandomForest",
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "CV_RMSE": cv_rmse_mean,
    }
])
metrics_df

,model,MAE,RMSE,R2,CV_RMSE
0,RandomForest,3.698578e+09,5.230566e+09,0.762759,5.656034e+09


## 3) Luu model RandomForest vao backend

In [8]:
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
dump(pipeline, MODEL_PATH)

print("Model su dung: RandomForest")
print("Da luu model tai:", MODEL_PATH)

Model su dung: RandomForest
Da luu model tai: C:\Users\asus\Desktop\DuDoanGiaNha\webDemo\BackEnd\model1.joblib


In [9]:
# Test nhanh 1 mau de dam bao model chay duoc
sample = X_test.head(1)
sample_pred = pipeline.predict(sample)[0]
print("Du doan mau test:", float(sample_pred))

Du doan mau test: 13256125000.0
